### **BÀI TẬP**

Code Folder Structure:
```bash
face_comparison_project/
│
├── data/
│   ├── raw/                  # Chứa ảnh gốc
│   ├── preprocessed/         # Nơi lưu ảnh sau khi thực hiện tiền xử lý
│   └── results/              # Nơi lưu ảnh sau khi đã phát hiện khuôn mặt và vẽ bounding box
│
├── models/                   
│   └── yolov8n-face.pt                      # Tải từ HuggingFace
│
├── src/
│   ├── __init__.py
│   ├── data.py               # Chứa hàm đọc và ghi file ảnh
│   ├── preprocessing.py      # Chứa hàm tiền xử lý ảnh
│   └── detectors.py          # Chứa 2 Class cho 2 mô hình
│
├── requirements.txt
├── .gitignore
└── main.py                   # File chạy chính
```

Yêu cầu: Hoàn thiện nội dung các files có trong folder để triển khai bài toán Phát hiện khuôn mặt sử dụng 2 mô hình Haar Cascades và Yolov8. Lần lượt thực hiện các bước sau:
1. Tìm và tải 5 bức ảnh nhóm (có nhiều người) và lưu vào /data/raw, đặt tên theo cú pháp `img_{id}.jpg` với id từ 0 đến 4.
2. Chạy đoạn code bên dưới để tải pretrained yolov8-face model về máy và lưu vào /models.
3. Hoàn thiện các file trong folder /src theo hướng dẫn sau:

**a. Hoàn thiện File src/data_io.py (Quản lý File & Dữ liệu)**
- Mục tiêu: Xây dựng một module chuyên biệt chịu trách nhiệm giao tiếp với ổ cứng. Tuân thủ nguyên tắc: Không để thuật toán AI can thiệp trực tiếp vào việc đọc/ghi file.
- Yêu cầu chi tiết cho từng hàm:
    - Hàm get_image_files(input_dir):
        - Kiểm tra xem thư mục input_dir có tồn tại hay không. Nếu không, hãy dùng mã lệnh tự động tạo thư mục đó và trả về một danh sách rỗng.
        - Duyệt qua tất cả các file trong thư mục. Chỉ thu thập những file có đuôi mở rộng là .jpg, .jpeg, hoặc .png (nhớ xử lý chữ hoa/chữ thường).
        - Trả về một list chứa đường dẫn đầy đủ của các file ảnh hợp lệ đó.
    - Hàm load_image(filepath):
        - Sử dụng hàm của OpenCV để nạp ảnh từ đường dẫn vào RAM.
        - Yêu cầu bắt buộc: OpenCV sẽ không báo lỗi nếu đường dẫn sai mà chỉ ngầm trả về None. Viết câu lệnh if kiểm tra: nếu ảnh bị None, hãy sử dụng lệnh raise ValueError() để chặn hệ thống lại ngay lập tức.
    - Hàm save_image(img, output_dir, filename):
        - Sử dụng thư viện hệ điều hành (os) để tạo thư mục output_dir một cách an toàn (không báo lỗi nếu thư mục đã tồn tại).
        - Ghép nối output_dir và filename thành một đường dẫn hoàn chỉnh.
        - Lưu ma trận ảnh (img) xuống ổ cứng tại đường dẫn vừa ghép.

**b. Hoàn thiện File src/preprocessing.py (Tiền xử lý)**
- Mục tiêu: Viết các bộ lọc chuẩn hóa dữ liệu đầu vào.
- Yêu cầu chi tiết cho hàm resize_image(img, max_size=1024):
    - Lấy ra kích thước chiều cao (h) và chiều rộng (w) của bức ảnh truyền vào.
    - Tìm ra cạnh dài nhất của bức ảnh. Nếu cạnh dài nhất này nhỏ hơn hoặc bằng max_size (1024 pixel), hãy return ngay bức ảnh gốc (không cần xử lý gì thêm).
    - Nếu cạnh dài nhất lớn hơn max_size, bạn phải tính toán tỷ lệ thu nhỏ sao cho cạnh lớn nhất bị ép về đúng 1024 pixel, và cạnh còn lại thu nhỏ theo đúng tỷ lệ đó (để mặt người không bị dẹt hay bóp méo).
    - Sử dụng lệnh thu nhỏ của OpenCV (gợi ý: sử dụng nội suy cv2.INTER_LINEAR cho chất lượng tốt nhất) và trả về bức ảnh đã thu nhỏ.

**c. Hoàn thiện File src/detectors.py**
- Mục tiêu: Đóng gói 2 thuật toán khác nhau vào 2 Lớp (Class) riêng biệt.
- Yêu cầu cho Class HaarFaceDetector:
    - Hàm khởi tạo __init__: Khởi tạo công cụ cv2.CascadeClassifier.
    - Hàm detect_and_draw(img):
        - Tạo một bản sao của ảnh gốc để vẽ lên đó (tránh làm hỏng ảnh hệ thống).
        - Chuyển bản sao này sang ảnh xám (Grayscale) vì thuật toán Haar yêu cầu như vậy.
        - Gọi hàm nhận diện. Với mỗi khuôn mặt tìm được, vẽ một hình chữ nhật MÀU ĐỎ (nhớ lại hệ màu BGR) có độ dày viền là 2 pixel.
        - Trả về: (bức_ảnh_đã_vẽ, số_lượng_khuôn_mặt).

- Yêu cầu cho Class YoloFaceDetector:
    - Hàm khởi tạo __init__: Nhận vào đường dẫn file .pt. Khởi tạo mô hình AI bằng thư viện YOLO của Ultralytics.
    - Hàm detect_and_draw(img):
        - Cũng tạo một bản sao của ảnh gốc.
        - Ném ảnh vào hàm .predict() của YOLO.
        - Trích xuất dữ liệu tọa độ hộp (boxes) từ kết quả. Dùng vòng lặp bóc tách tọa độ (x1, y1, x2, y2) ép kiểu về số nguyên, và vẽ hình chữ nhật MÀU XANH LÁ lên ảnh.
        - Trả về: (bức_ảnh_đã_vẽ, số_lượng_khuôn_mặt).

**d. Hoàn thiện File main.py**
- Mục tiêu: Đóng vai trò gọi các hàm ở 3 file trên lắp ghép lại thành một dây chuyền tự động từ A đến Z.
- Yêu cầu luồng thực thi:
    - Khởi tạo: Hãy khởi tạo cả 2 mô hình (Haar và YOLO) trước khi vòng lặp bắt đầu. Hãy dùng try...except bao bọc bước này để lỡ mô hình không nạp được thì in thông báo và dừng chương trình.
    - Đọc dữ liệu: Gọi hàm lấy danh sách ảnh từ thư mục data/raw. Nếu danh sách trống, in cảnh báo và thoát.
    - Vòng lặp xử lý: Duyệt qua từng đường dẫn ảnh:
        - Đọc bức ảnh lên.
        - Đưa ảnh vào hàm giới hạn kích thước (resize).
        - Lưu lại bức ảnh đã tiền xử lý này vào thư mục data/preprocessed làm tư liệu.
    - Thử nghiệm Haar: Lấy ảnh đã tiền xử lý ném vào mô hình Haar. Lưu ảnh kết quả vào data/results với tiền tố haar_.
    - Thử nghiệm YOLO: Lấy ảnh đã tiền xử lý ném vào mô hình YOLO. Lưu ảnh kết quả vào data/results với tiền tố yolo_.
    - In ra màn hình console số lượng khuôn mặt tìm được của mỗi mô hình để so sánh. (Bọc toàn bộ vòng lặp xử lý 1 ảnh trong try...except để nếu 1 ảnh bị lỗi, hệ thống không sập mà nhảy sang xử lý ảnh tiếp theo).

In [4]:
import os
from huggingface_hub import hf_hub_download

# 1. Create the 'models' directory if it doesn't exist
os.makedirs("./face-detection/models", exist_ok=True)

# 2. Download the model directly into the 'models' directory
hf_hub_download(
    repo_id="arnabdhar/YOLOv8-Face-Detection", 
    filename="model.pt",
    local_dir="./face-detection/models",
    local_dir_use_symlinks=False
)

# 3. Define current path and your desired new path
old_path = os.path.join("./face-detection/models", "model.pt")
new_path = os.path.join("./face-detection/models", "yolov8-face.pt")

# 4. Rename the file safely
if os.path.exists(old_path):
    if os.path.exists(new_path):
        os.remove(new_path)  # Overwrites existing file if you run this script twice
    os.rename(old_path, new_path)
    print(f"Success! Model saved to: {new_path}")


c:\Users\phaml\anaconda3\envs\gpu\lib\site-packages\huggingface_hub\file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.pt:   0%|          | 0.00/6.25M [00:00<?, ?B/s]

Success! Model saved to: ./face-detection/models\yolov8-face.pt
